In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import LabelEncoder 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- 1. Fungsi Pra-pemrosesan Data Konsisten ---
def preprocess_data(df):
    """
    Menerapkan langkah pra-pemrosesan yang sama pada data train atau test.
    """
    features_to_use = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
    X = df[features_to_use].copy()
    
    # 1. Menangani Nilai yang Hilang (Imputasi)
    imputer = SimpleImputer(strategy='median')
    X[['Age', 'Fare']] = imputer.fit_transform(X[['Age', 'Fare']])
    
    # 2. Mengubah Fitur Kategorikal (Encoding)
    le = LabelEncoder()
    X['Sex_encoded'] = le.fit_transform(X['Sex'])
    
    # Pilih fitur numerik final
    X_final = X[['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare']]
    
    return X_final

# --- 2. Memuat dan Mempersiapkan Data ---
train_file_path = '/kaggle/input/titanic/train.csv' 
test_file_path = '/kaggle/input/titanic/test.csv'

train_data = pd.read_csv(train_file_path)
test_data = pd.read_csv(test_file_path)

y = train_data.Survived

X_train_processed = preprocess_data(train_data)
X_test_processed = preprocess_data(test_data)

X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train_processed, y, random_state=1, test_size=0.2
)

# --- 3. Pelatihan Model ---
rf_model_titanic = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
rf_model_titanic.fit(X_train_model, y_train_model)

val_predictions = rf_model_titanic.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)

print("--- Hasil Evaluasi Validasi ---")
print("Validation Accuracy: {:.4f}".format(val_accuracy))
print("-" * 30)

# --- 4. Membuat Prediksi Final dan Submission ---
final_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
final_model.fit(X_train_processed, y)

test_predictions = final_model.predict(X_test_processed)

submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': test_predictions
})

submission['Survived'] = submission['Survived'].astype(int)
submission.to_csv('titanic_submission.csv', index=False)
print("Berkas submission 'titanic_submission.csv' berhasil dibuat.")
print("-" * 30)


# --- 5. Output: Diagram Sebaran Kelangsungan Hidup ---
# Menambahkan visualisasi sebaran kelangsungan hidup seperti pada gambar
print("--- DIAGRAM: Sebaran Kelangsungan Hidup ---")

plt.figure(figsize=(7, 5))
# Membuat barplot dari value_counts() kolom 'Survived'
sns.countplot(x=train_data['Survived'], palette=['salmon', 'mediumaquamarine']) # Menggunakan palette seperti gambar
plt.title('Sebaran Kelangsungan Hidup (1=Selamat, 0=Tidak)')
plt.xlabel('Kelangsungan Hidup')
plt.ylabel('Jumlah Orang')
plt.xticks(ticks=[0, 1], labels=['Tidak (0)', 'Selamat (1)']) # Memberi label lebih jelas pada sumbu X
plt.show()

print("\nInterpretasi Diagram Sebaran Kelangsungan Hidup:")
print(f"* Dari {len(train_data)} penumpang di data pelatihan:")
print(f"  - **{train_data['Survived'].value_counts()[0]}** orang tidak selamat.")
print(f"  - **{train_data['Survived'].value_counts()[1]}** orang selamat.")
print("* Terlihat bahwa jumlah orang yang tidak selamat lebih banyak daripada yang selamat.")
print("-" * 30)


# --- 6. Output: Diagram Pentingnya Fitur (Feature Importance) ---
# Mengganti nomor bagian karena ada tambahan diagram
print("--- DIAGRAM: Pentingnya Fitur (Feature Importance) ---")

feature_importances = final_model.feature_importances_
features = X_train_processed.columns

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Visualisasi
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Pentingnya Fitur untuk Prediksi Kelangsungan Hidup Titanic')
plt.xlabel('Nilai Kepentingan')
plt.ylabel('Fitur')
plt.show()

print("\nInterpretasi Diagram Pentingnya Fitur:")
print(f"* Fitur **{importance_df.iloc[0]['Feature']}** adalah yang paling penting (Importance: {importance_df.iloc[0]['Importance']:.3f}).")
print(f"* Ini menunjukkan bahwa **Jenis Kelamin (Sex)** memainkan peran terbesar dalam peluang bertahan hidup, diikuti oleh {importance_df.iloc[1]['Feature']} (Kelas Tiket).")
print("* Fitur yang kurang penting seperti Parch dan SibSp memiliki nilai Importance yang rendah, tetapi masih berkontribusi pada model.")
print("-" * 30)